# Q-Mini Add/Sub v2 — Scaled QAT English Training

This notebook builds the **new independent model line** inside the Q-mini repository. It does not load or modify the old Q-Mini Light model, its checkpoints, tokenizer or datasets.

Important: data preparation, tokenizer building, pretraining and SFT are all **OFF by default**. You can import your own already-reviewed English dataset from Drive without downloading a dataset, or explicitly choose the pinned optional corpus. CPU is the safe default; choose CUDA only when you intentionally want to use a GPU. The notebook checks out the repository's main branch.

Data sources: pinned Simple English Wikipedia (2026-03-01 dump) for language pretraining, reviewed/non-synthetic OASST1 conversations and human-authored Dolly instruction pairs. TinyStories and model-generated synthetic corpora are excluded. DailyDialog is excluded because its license has non-commercial/share-alike restrictions.

The 5-symbol weight alphabet needs 3-bit storage. The training graph uses floating-point optimizer arithmetic; the complete multiply-free inference runtime is still under development. Do not use the training graph as a deployed inference engine.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/lion36508/Q-mini-light-0.7M.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/content/Q-mini-light-0.7M')

if not (REPO_DIR / '.git').is_dir():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)

REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
print('Repository branch:', REPO_BRANCH)
print('Repository commit:', REPO_COMMIT)

DRIVE_ROOT = Path('/content/drive/MyDrive/Q mini 2 light')
VERSION_ROOT = DRIVE_ROOT / 'trainability_addsub_main_v1'

# Workspace safety:
# - Default mode refuses to silently reuse non-empty data/checkpoint folders.
# - Fresh mode writes to a new isolated folder and never deletes older runs.
# - Set RESUME_EXISTING_RUN=True only when intentionally continuing that exact run.
FORCE_FRESH_RUN = False
FRESH_RUN_ID = ''  # Example: 'retrain-2026-11-26-a'; required when FORCE_FRESH_RUN=True.
RESUME_EXISTING_RUN = False

# Install data/tokenizer tools without replacing Colab's preinstalled PyTorch.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.24', 'tokenizers>=0.15'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-q', '-e', str(REPO_DIR)], check=True)

from qmini_addsub.checkpoints import resolve_checkpoint
from qmini_addsub.workspace import resolve_workspace_root

WORKSPACE_ROOT = resolve_workspace_root(
    VERSION_ROOT,
    force_fresh=FORCE_FRESH_RUN,
    run_id=FRESH_RUN_ID or None,
    resume_existing=RESUME_EXISTING_RUN,
)
DATA_DIR = WORKSPACE_ROOT / 'data'
CACHE_DIR = WORKSPACE_ROOT / 'cache'
USER_DATASET_DIR = WORKSPACE_ROOT / 'selected_dataset'
ARTIFACT_DIR = WORKSPACE_ROOT / 'artifacts'
PRETRAIN_DIR = WORKSPACE_ROOT / 'runs' / 'pretrain'
SFT_DIR = WORKSPACE_ROOT / 'runs' / 'sft'
for folder in (DATA_DIR, CACHE_DIR, ARTIFACT_DIR, PRETRAIN_DIR, SFT_DIR, USER_DATASET_DIR):
    folder.mkdir(parents=True, exist_ok=True)


import torch
print('PyTorch:', torch.__version__)
print('Available CUDA:', torch.cuda.is_available())
print('New model Drive root:', DRIVE_ROOT)
print('Selected Add/Sub workspace:', WORKSPACE_ROOT)


## Source-license approval

Before downloading, review the dataset cards and obligations: Simple English Wikipedia (CC BY-SA / GFDL metadata), OASST1 (Apache-2.0), Dolly 15k (CC BY-SA 3.0). Share-alike and attribution obligations may affect redistribution. This notebook makes no legal determination.

In [ ]:
# Select ONE data path only after checking the dataset and its license.
PREPARE_PINNED_CORPUS = False
IMPORT_USER_DATA = False
ACCEPT_SOURCE_LICENSES = False

# For a user-selected dataset, place these files in the Drive folder printed above.
USER_PRETRAIN_TRAIN = USER_DATASET_DIR / 'train.txt'
USER_PRETRAIN_VALIDATION = USER_DATASET_DIR / 'validation.txt'
USER_PRETRAIN_TEST = USER_DATASET_DIR / 'test.txt'
USER_PRETRAIN_SOURCE = ''   # Dataset name / revision / source URL.
USER_PRETRAIN_LICENSE = ''  # Exact license shown by the source.

# Optional supervised assistant data. Leave both paths as None until selected.
USER_SFT_TRAIN = None
USER_SFT_VALIDATION = None
USER_SFT_TEST = None
USER_SFT_SOURCE = ''
USER_SFT_LICENSE = ''

if PREPARE_PINNED_CORPUS and IMPORT_USER_DATA:
    raise ValueError('Choose either the pinned corpus or your own local dataset, not both.')

if PREPARE_PINNED_CORPUS:
    if not ACCEPT_SOURCE_LICENSES:
        raise RuntimeError('Review the pinned sources and their licenses, then set ACCEPT_SOURCE_LICENSES=True.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=0.28', 'pyarrow>=15.0'], check=True)
    prepare_command = [
        sys.executable, 'scripts/prepare_qmini_addsub_data.py',
        '--output', str(DATA_DIR),
        '--cache', str(CACHE_DIR),
        '--accept-source-licenses',
    ]
    subprocess.run(prepare_command, cwd=REPO_DIR, check=True)
    print('Optional pinned English dataset prepared. No model training has started.')

elif IMPORT_USER_DATA:
    required = [USER_PRETRAIN_TRAIN, USER_PRETRAIN_VALIDATION, USER_PRETRAIN_TEST]
    missing = [str(path) for path in required if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Copy the selected train/validation/test text files into Drive first: ' + ', '.join(missing))
    if not USER_PRETRAIN_SOURCE.strip() or not USER_PRETRAIN_LICENSE.strip():
        raise ValueError('Enter the selected dataset source/revision and its license.')
    import_command = [
        sys.executable, 'scripts/import_qmini_addsub_dataset.py',
        '--pretrain-train', str(USER_PRETRAIN_TRAIN),
        '--pretrain-validation', str(USER_PRETRAIN_VALIDATION),
        '--pretrain-test', str(USER_PRETRAIN_TEST),
        '--pretrain-source', USER_PRETRAIN_SOURCE,
        '--pretrain-license', USER_PRETRAIN_LICENSE,
        '--output', str(DATA_DIR),
        '--confirm-rights-and-human-data',
        '--confirm-english-data',
    ]
    if any(value is not None for value in (USER_SFT_TRAIN, USER_SFT_VALIDATION, USER_SFT_TEST)):
        if any(value is None for value in (USER_SFT_TRAIN, USER_SFT_VALIDATION, USER_SFT_TEST)):
            raise ValueError('Provide SFT train, validation and test files together.')
        if not USER_SFT_SOURCE.strip() or not USER_SFT_LICENSE.strip():
            raise ValueError('Enter the SFT dataset source and license.')
        import_command += [
            '--sft-train', str(USER_SFT_TRAIN),
            '--sft-validation', str(USER_SFT_VALIDATION),
            '--sft-test', str(USER_SFT_TEST),
            '--sft-source', USER_SFT_SOURCE,
            '--sft-license', USER_SFT_LICENSE,
        ]
    subprocess.run(import_command, cwd=REPO_DIR, check=True)
    print('Your selected local English data has been imported. No dataset was downloaded.')

else:
    print('DATA PREPARATION IS OFF. No dataset has been downloaded or imported.')
    print('After choosing a dataset, set IMPORT_USER_DATA=True and fill the Drive paths/source/license above.')


In [ ]:
BUILD_TOKENIZER_AND_STREAMS = False
TOKENIZER_PATH = ARTIFACT_DIR / 'tokenizer.json'
MODEL_CONFIG_PATH = ARTIFACT_DIR / 'model_config.json'

if BUILD_TOKENIZER_AND_STREAMS:
    if not (DATA_DIR / 'pretrain_train.txt').is_file() or not (DATA_DIR / 'dataset_manifest.json').is_file():
        raise FileNotFoundError('Prepare/import your selected dataset first.')
    subprocess.run([
        sys.executable, 'scripts/train_qmini_addsub_tokenizer.py',
        '--data-dir', str(DATA_DIR),
        '--output', str(TOKENIZER_PATH),
        '--vocab-size', '1024',
    ], cwd=REPO_DIR, check=True)

    subprocess.run([
        sys.executable, 'scripts/encode_qmini_addsub_streams.py',
        '--data-dir', str(DATA_DIR),
        '--tokenizer', str(TOKENIZER_PATH),
    ], cwd=REPO_DIR, check=True)
    print('Tokenizer and verified train/validation/test token streams are ready.')
else:
    print('TOKENIZER BUILD AND STREAM ENCODING ARE OFF.');
    print('Set BUILD_TOKENIZER_AND_STREAMS=True only after reviewing and importing your dataset.')


## Stage 1 — base English pretraining

Start from random initialization. This stage writes checkpoints only under the separate Add/Sub v1 Drive folder. The default is CPU and a small trial run. Raise the step count only after reviewing held-out loss and available compute.

In [ ]:
START_PRETRAIN = False  # Enable only after the selected English data is ready.
DEVICE = 'cpu'          # Safe default. Set to 'cuda' only intentionally.
if DEVICE.startswith('cuda') and not torch.cuda.is_available():
    raise RuntimeError('CUDA was selected, but this Colab runtime has no visible CUDA GPU. Select Runtime > Change runtime type > a GPU runtime, then reconnect.')
PRETRAIN_STEPS = 1000
BATCH_SIZE = 2 if DEVICE == 'cpu' else 8
SEQUENCE_LENGTH = 256
PRETRAIN_NUM_THREADS = 0  # 0 keeps Colab's PyTorch default; set a positive integer to tune CPU intra-op threads.
if isinstance(PRETRAIN_NUM_THREADS, bool) or not isinstance(PRETRAIN_NUM_THREADS, int) or PRETRAIN_NUM_THREADS < 0:
    raise ValueError('PRETRAIN_NUM_THREADS must be a non-negative integer.')
PRETRAIN_RESUME = None  # Optional explicit checkpoint; None auto-selects the newest compatible candidate.
PRETRAIN_EXTEND_SCHEDULE = False  # True only when intentionally increasing a saved run's optimizer-step horizon.

if START_PRETRAIN:
    required_files = [DATA_DIR / 'pretrain_train.u16', DATA_DIR / 'pretrain_validation.u16', TOKENIZER_PATH, MODEL_CONFIG_PATH, DATA_DIR / 'dataset_manifest.json']
    missing = [str(path) for path in required_files if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Pretraining cannot start; missing selected dataset/tokenizer artifacts: ' + ', '.join(missing))

    resume_path, resume_step = resolve_checkpoint(
        PRETRAIN_DIR,
        prefix='checkpoint',
        metrics_filename='metrics.jsonl',
        explicit=PRETRAIN_RESUME,
    )
    if resume_path is not None and resume_step >= PRETRAIN_STEPS:
        print(f'Pretraining already reached step {resume_step}; target is {PRETRAIN_STEPS}. Skipping without rewriting checkpoints.')
    else:
        command = [
            sys.executable, '-m', 'qmini_addsub.train',
            '--train-bin', str(DATA_DIR / 'pretrain_train.u16'),
            '--valid-bin', str(DATA_DIR / 'pretrain_validation.u16'),
            '--tokenizer', str(TOKENIZER_PATH),
            '--config', str(MODEL_CONFIG_PATH),
            '--output', str(PRETRAIN_DIR),
            '--device', DEVICE,
            '--steps', str(PRETRAIN_STEPS),
            '--batch-size', str(BATCH_SIZE),
            '--sequence-length', str(SEQUENCE_LENGTH),
        ]
        if PRETRAIN_NUM_THREADS > 0:
            command += ['--threads', str(PRETRAIN_NUM_THREADS)]
        if resume_path is not None:
            command += ['--resume', str(resume_path)]
            print(f'Automatically resuming pretraining from step {resume_step}: {resume_path}')
        if PRETRAIN_EXTEND_SCHEDULE:
            command += ['--extend-schedule']
        else:
            print('No pretraining checkpoint exists; starting a new run in the isolated Add/Sub folder.')
        subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print('Pretraining is OFF. Enable it only after reviewing the selected English data.')


## Stage 2 — supervised instruction fine-tuning

Only run after Stage 1 has produced a non-zero-step Add/Sub v1 checkpoint and its validation loss has been inspected. The SFT trainer rejects legacy Q2-v2 checkpoints and random step-zero artifacts.

In [ ]:
START_SFT = False
SFT_STEPS = 500
SFT_SAVE_EVERY = 100
SFT_DEVICE = DEVICE
PRETRAIN_CHECKPOINT = None  # Optional explicit Add/Sub pretraining checkpoint.
SFT_RESUME = None            # None auto-selects the newest Add/Sub SFT checkpoint.
SFT_EXTEND_SCHEDULE = False  # True only when intentionally increasing the saved SFT optimizer-step horizon.

if START_SFT:
    required_sft_files = [TOKENIZER_PATH, DATA_DIR / 'sft_train.jsonl', DATA_DIR / 'sft_validation.jsonl', DATA_DIR / 'sft_test.jsonl', DATA_DIR / 'dataset_manifest.json']
    missing = [str(path) for path in required_sft_files if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('SFT cannot start; import reviewed human-authored SFT splits first: ' + ', '.join(missing))

    PRETRAIN_CHECKPOINT, pretrain_step = resolve_checkpoint(
        PRETRAIN_DIR,
        prefix='checkpoint',
        metrics_filename='metrics.jsonl',
        explicit=PRETRAIN_CHECKPOINT,
    )
    if PRETRAIN_CHECKPOINT is None:
        raise FileNotFoundError('No Add/Sub pretraining checkpoint found. Finish Stage 1 first.')
    print(f'Using Add/Sub pretraining checkpoint step {pretrain_step}: {PRETRAIN_CHECKPOINT}')

    sft_resume_path, sft_resume_step = resolve_checkpoint(
        SFT_DIR,
        prefix='sft-checkpoint',
        metrics_filename='sft_metrics.jsonl',
        explicit=SFT_RESUME,
    )
    if sft_resume_path is not None and sft_resume_step >= SFT_STEPS:
        print(f'SFT already reached step {sft_resume_step}; target is {SFT_STEPS}. Skipping without rewriting checkpoints.')
    else:
        command = [
            sys.executable, '-m', 'qmini_addsub.sft',
            '--base-checkpoint', str(PRETRAIN_CHECKPOINT),
            '--tokenizer', str(TOKENIZER_PATH),
            '--train-jsonl', str(DATA_DIR / 'sft_train.jsonl'),
            '--valid-jsonl', str(DATA_DIR / 'sft_validation.jsonl'),
            '--test-jsonl', str(DATA_DIR / 'sft_test.jsonl'),
            '--output', str(SFT_DIR),
            '--device', SFT_DEVICE,
            '--steps', str(SFT_STEPS),
            '--save-every', str(SFT_SAVE_EVERY),
        ]
        if sft_resume_path is not None:
            command += ['--resume', str(sft_resume_path)]
            print(f'Automatically resuming assistant SFT from step {sft_resume_step}: {sft_resume_path}')
        if SFT_EXTEND_SCHEDULE:
            command += ['--extend-schedule']
        else:
            print('No SFT checkpoint exists; starting a new SFT run from the selected base checkpoint.')
        subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print('SFT is OFF. Enable only after real-data pretraining and SFT split review.')


In [ ]:
print('Add/Sub v1 data:', DATA_DIR)
print('Add/Sub v1 tokenizer:', TOKENIZER_PATH)
print('Pretraining checkpoints:', PRETRAIN_DIR)
print('SFT checkpoints:', SFT_DIR)
print('Legacy model and legacy Drive folder were not selected by this notebook.')
print('Full integer-only mobile inference is not yet complete.')

RUN_HELDOUT_EVAL = False
HELDOUT_EVAL_CHECKPOINT = None
HELDOUT_EVAL_DIR = WORKSPACE_ROOT / 'evaluation' / 'heldout'
if RUN_HELDOUT_EVAL:
    required = [TOKENIZER_PATH, DATA_DIR / 'sft_test.jsonl', DATA_DIR / 'dataset_manifest.json']
    missing = [str(path) for path in required if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Held-out SFT evaluation cannot start; missing: ' + ', '.join(missing))
    HELDOUT_EVAL_CHECKPOINT, heldout_step = resolve_checkpoint(
        SFT_DIR,
        prefix='sft-checkpoint',
        metrics_filename='sft_metrics.jsonl',
        explicit=HELDOUT_EVAL_CHECKPOINT,
    )
    if HELDOUT_EVAL_CHECKPOINT is None:
        raise FileNotFoundError('No SFT checkpoint exists for assistant evaluation.')
    print(f'Evaluating SFT checkpoint step {heldout_step}: {HELDOUT_EVAL_CHECKPOINT}')
    subprocess.run([
        sys.executable, '-m', 'qmini_addsub.evaluate',
        '--checkpoint', str(HELDOUT_EVAL_CHECKPOINT),
        '--tokenizer', str(TOKENIZER_PATH),
        '--test-jsonl', str(DATA_DIR / 'sft_test.jsonl'),
        '--dataset-manifest', str(DATA_DIR / 'dataset_manifest.json'),
        '--output', str(HELDOUT_EVAL_DIR),
        '--device', DEVICE,
    ], cwd=REPO_DIR, check=True)
else:
    print('Held-out assistant evaluation is OFF. Enable only after SFT.')

# Optional fail-closed quality gate. This never trains or changes model weights.
RUN_READINESS_GATE = False
READINESS_REPORT_PATH = None
if RUN_READINESS_GATE:
    metrics_path = SFT_DIR / 'sft_metrics.jsonl'
    heldout_summary_path = HELDOUT_EVAL_DIR / 'heldout_summary.json'
    missing = [str(path) for path in (metrics_path, heldout_summary_path) if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError(
            'Readiness gate needs SFT metrics and a completed held-out evaluation report: '
            + ', '.join(missing)
        )
    readiness_checkpoint, readiness_step = resolve_checkpoint(
        SFT_DIR,
        prefix='sft-checkpoint',
        metrics_filename='sft_metrics.jsonl',
        explicit=None,
    )
    if readiness_checkpoint is None:
        raise FileNotFoundError('No SFT checkpoint exists for readiness evaluation.')
    READINESS_REPORT_PATH = ARTIFACT_DIR / f'addsub-readiness-step-{readiness_step}.json'
    readiness_result = subprocess.run([
        sys.executable, '-m', 'qmini_addsub.readiness',
        '--metrics', str(metrics_path),
        '--checkpoint', str(readiness_checkpoint),
        '--heldout-summary', str(heldout_summary_path),
        '--output', str(READINESS_REPORT_PATH),
    ], cwd=REPO_DIR, check=False)
    if readiness_result.returncode not in (0, 2) or not READINESS_REPORT_PATH.is_file():
        raise RuntimeError('Add/Sub readiness gate failed to produce a valid report.')
    import json as _readiness_json
    readiness_report = _readiness_json.loads(READINESS_REPORT_PATH.read_text(encoding='utf-8'))
    print('Add/Sub readiness:', readiness_report['status'])
    print('Readiness report:', READINESS_REPORT_PATH)
    if not readiness_report['ready']:
        print('NOT READY: do not promote or present this checkpoint as a usable assistant.')
    else:
        print('READY FOR HUMAN REVIEW only; this is not a certification of assistant quality.')
else:
    print('Readiness gate is OFF. Enable after held-out evaluation if you want a promotion check.')

EXPORT_PACKED_WEIGHTS = False
EXPORT_CHECKPOINT = None  # Optional explicit Add/Sub pretraining or SFT checkpoint.
EXPORT_PATH = ARTIFACT_DIR / 'qmini_addsub_model.qma'

if EXPORT_PACKED_WEIGHTS:
    if EXPORT_CHECKPOINT is None:
        EXPORT_CHECKPOINT, export_step = resolve_checkpoint(
            SFT_DIR,
            prefix='sft-checkpoint',
            metrics_filename='sft_metrics.jsonl',
            explicit=None,
        )
        if EXPORT_CHECKPOINT is None:
            EXPORT_CHECKPOINT, export_step = resolve_checkpoint(
                PRETRAIN_DIR,
                prefix='checkpoint',
                metrics_filename='metrics.jsonl',
                explicit=None,
            )
        if EXPORT_CHECKPOINT is None:
            raise FileNotFoundError('No trained Add/Sub checkpoint is available to export.')
        print(f'Exporting newest checkpoint at step {export_step}: {EXPORT_CHECKPOINT}')
    subprocess.run([
        sys.executable, '-m', 'qmini_addsub.export',
        '--checkpoint', str(EXPORT_CHECKPOINT),
        '--output', str(EXPORT_PATH),
    ], cwd=REPO_DIR, check=True)
else:
    print('Packed weight export is OFF. Enable only after training.')
